In [ ]:
import warnings
warnings.filterwarnings('ignore')

from datetime import datetime

import numpy as np
import pandas as pd
from scipy.stats.mstats import winsorize
import matplotlib.pyplot as plt

In [ ]:
# Loading in saved DataFrame from other script
df = pd.read_pickle('russ_250_df.pkl')

In [ ]:
# To evaluate MultiIndex data throughout the process
idx = pd.IndexSlice

In [ ]:
# Rearranging columns
col_order = ['close', 'market_cap', 'dividends_ttm', 'buybacks_ttm', 'total_payout_ttm', 'sector']
df = df[col_order]

In [ ]:
# Configuration class to define parameters used throughout the code
class Config:
    BACKTEST_START = '2010-01-01'
    BACKTEST_END = '2025-12-31'

    LOOKBACK = 252         
    SIGNAL_WEIGHTS = {'sy_level': 0.50, 'sy_growth': 0.50}
    
    # Paramter for exponential tilt function
    TILT_LAMBDA = 2             

    # Single-stock and sector caps
    MIN_WEIGHT = 0.0010             
    MAX_WEIGHT = 0.08                
    SECTOR_CAP_MULTIPLE = 2.0       

    # Semi-annual rebalance
    REBALANCE_FREQ = '6ME'           

    # One-way turnover cost in basis points
    TC_BPS = 15

In [ ]:
# Converting each feature into wide panel format for easier calculations later in code
def pivot_panel(df: pd.DataFrame, column: str) -> pd.DataFrame:
    return df[column].unstack(level='RIC').sort_index()


close = pivot_panel(df, 'close')
market_cap = pivot_panel(df, 'market_cap')
dividends = pivot_panel(df, 'dividends_ttm')
buybacks = pivot_panel(df, 'buybacks_ttm')
total_payout = pivot_panel(df, 'total_payout_ttm')

sector_panel = pivot_panel(df, 'sector')
sectors = sector_panel.ffill().iloc[-1]

In [ ]:
# Calculating shareholder yield and sharehold yield growth metrics
sy_level = total_payout / market_cap.replace(0, np.nan)
sy_level = sy_level.replace([np.inf, -np.inf], np.nan)

prior_payout_1 = total_payout.shift(Config.LOOKBACK)
prior_payout_2 = total_payout.shift(Config.LOOKBACK * 2)
prior_payout_3 = total_payout.shift(Config.LOOKBACK * 3)

# Adding in epsilon to avoid division by zero - using arc tangent function to compress any outliers 
epsilon = 1e6
growth1 = np.arctan((total_payout - prior_payout_1) / (prior_payout_1.abs() + epsilon))
growth2 = np.arctan((prior_payout_1 - prior_payout_2) / (prior_payout_2.abs() + epsilon))
growth3 = np.arctan((prior_payout_2 - prior_payout_3) / (prior_payout_3.abs() + epsilon))

sy_growth = (growth1 + growth2 + growth3) / 3
sy_growth = sy_growth.replace([np.inf, -np.inf], np.nan)

In [ ]:
# Performing cross-sectional standardization of stocks
def cross_sectional_zscore(signal: pd.Series):
    valid = signal.dropna()
    mu = valid.mean()
    sigma = valid.std()

    z = (valid - mu) / sigma
    return z.reindex(signal.index)

# Putting together the composite score
def build_composite_score(z_sy: pd.Series, z_sy_growth: pd.Series, weights: dict = Config.SIGNAL_WEIGHTS):
    composite = (
        weights['sy_level'] * z_sy.fillna(0)
        + weights['sy_growth'] * z_sy_growth.fillna(0)
    )
    # Penalizing stocks missing both signals
    both_missing = z_sy.isna() & z_sy_growth.isna()
    composite[both_missing] = np.nan
    return composite

In [ ]:
# Converting composite scores into portfolio weights
def compute_index_weights(composite: pd.Series, mktcap: pd.Series, sectors: pd.Series):
    valid = composite.dropna().index.intersection(mktcap.dropna().index)
    comp = composite.loc[valid]
    mc = mktcap.loc[valid]
    sec = sectors.reindex(valid)

    mc_weights = mc / mc.sum()
    tilt_factor = np.exp(Config.TILT_LAMBDA * comp)
    raw_weights = mc_weights * tilt_factor

    if raw_weights.sum() == 0:
        return pd.Series(dtype=float)
    
    weights = raw_weights / raw_weights.sum()

    # Capping weights of any single holding
    weights = weights.clip(upper=Config.MAX_WEIGHT)
    weights = weights[weights >= Config.MIN_WEIGHT]
    if weights.sum() == 0:
        return pd.Series(dtype=float)
    weights = weights / weights.sum()

    # Capping sectors at a multiple of the benchmark weighting
    if Config.SECTOR_CAP_MULTIPLE is not None and sec.notna().any():
        benchmark_sector_wt = mc.groupby(sec).sum() / mc.sum()
        sector_caps = benchmark_sector_wt * Config.SECTOR_CAP_MULTIPLE

        # Five iterations should be enough for sector cap adjustments
        for _ in range(5):  
            adjusted = False
            for sector in sec.dropna().unique():
                mask = sec.reindex(weights.index) == sector
                sector_wt = weights[mask].sum()
                cap = sector_caps.get(sector, 1.0)
                if sector_wt > cap and sector_wt > 0:
                    weights[mask] *= (cap / sector_wt)
                    adjusted = True
            weights = weights / weights.sum()
            if not adjusted:
                break

    return weights

In [ ]:
# Putting together list of rebalance dates
def build_rebalance_dates(start: str, end: str, freq: str, all_trading_days: pd.DatetimeIndex):
    period_ends = pd.date_range(start=start, end=end, freq=freq)
    rebal_dates = []
    
    # Snapping to the nearest trading day on or before the period end
    for pe in period_ends:
        eligible = all_trading_days[all_trading_days <= pe]
        if len(eligible) > 0:
            rebal_dates.append(eligible[-1])
    return pd.DatetimeIndex(sorted(set(rebal_dates)))


trading_days = close.index
rebal_dates = build_rebalance_dates(
    Config.BACKTEST_START,
    Config.BACKTEST_END,
    Config.REBALANCE_FREQ,
    trading_days,
)

In [ ]:
# Building weights for each rebalance date
weights_history = {}
composite_history = {}

for rebal_date in rebal_dates:
    sy_xsec = sy_level.loc[rebal_date]
    g_xsec = sy_growth.loc[rebal_date]
    mc_xsec = market_cap.loc[rebal_date]

    z_sy = cross_sectional_zscore(sy_xsec)
    z_g = cross_sectional_zscore(g_xsec)
    composite = build_composite_score(z_sy, z_g)

    weights = compute_index_weights(composite, mc_xsec, sectors)

    if len(weights) > 0:
        weights_history[rebal_date] = weights
        composite_history[rebal_date] = composite

print(f'Latest rebalance ({list(weights_history.keys())[-1].date()}):')
latest_w = weights_history[list(weights_history.keys())[-1]]
print(f'Max weight: {latest_w.max():.2%}')
print(f'Min weight: {latest_w.min():.2%}')
print(f'Sum check: {latest_w.sum():.2f}')

In [ ]:
# Putting together view of top holdings
latest_date = list(weights_history.keys())[-1]
latest_weights = weights_history[latest_date]
latest_composite = composite_history[latest_date]

print(f'Top 15 holdings as of {latest_date.date()}:')
print(f'{'RIC':<10} {'Weight':>8} {'Composite':>11} {'SY Level':>10} {'SY Growth':>11} {'Market Cap':>12} {'Sector':>9}')
print('-' * 80)
for RIC, wt in latest_weights.nlargest(15).items():
    sy_v = sy_level.loc[latest_date, RIC]
    g_v = sy_growth.loc[latest_date, RIC]
    mc_v = market_cap.loc[latest_date, RIC]
    cs = latest_composite.get(RIC, np.nan)
    sec = sectors.get(RIC, 'Unknown')
    print(f'{RIC:<10} {wt:>7.2%} {cs:>+11.2f} {sy_v:>12.2%} {g_v:>+10.2%} {mc_v/1e9:>12.2f}B {str(sec)[:24]:<35}')

In [ ]:
# Estimating transaction cost as one-way turnover * 15 bps 
def estimate_transaction_costs(old_weights: pd.Series, new_weights: pd.Series):
    all_RICs = old_weights.index.union(new_weights.index)
    old = old_weights.reindex(all_RICs, fill_value=0)
    new = new_weights.reindex(all_RICs, fill_value=0)
    one_way_turnover = (new - old).abs().sum() / 2
    return one_way_turnover * (Config.TC_BPS / 10000)

# Backtesting SY index and benchmark - returning the cumulative returns 
def run_backtest(weights_history: dict, price_panel: pd.DataFrame, benchmark_returns: pd.Series):
    rebal_dates = sorted(weights_history.keys())
    all_dates = price_panel.index.sort_values()

    daily_ret_records = []
    prev_weights = pd.Series(dtype=float)

    for i, rebal_date in enumerate(rebal_dates):
        target_weights = weights_history[rebal_date]
        tc = estimate_transaction_costs(prev_weights, target_weights)

        # Determining date range for this holding period
        start_idx = all_dates.get_loc(rebal_date)
        if i + 1 < len(rebal_dates):
            next_rebal = rebal_dates[i + 1]
            if next_rebal in all_dates:
                end_idx = all_dates.get_loc(next_rebal)
            else:
                end_idx = len(all_dates)
        else:
            end_idx = len(all_dates)

        period_dates = all_dates[start_idx:end_idx + 1]
        if len(period_dates) < 2:
            continue

        held_RICs = target_weights.index.intersection(price_panel.columns)
        period_prices = price_panel.loc[period_dates, held_RICs]
        daily_returns = period_prices.pct_change()

        # Initializing weights at start of period
        w = target_weights.reindex(held_RICs, fill_value=0)
        w = w / w.sum() if w.sum() > 0 else w

        for j, date in enumerate(period_dates[1:], 1):
            day_ret = daily_returns.loc[date].fillna(0)
            port_ret = (w * day_ret.reindex(w.index, fill_value=0)).sum()

            # Deducting transaction costs on rebalance day
            if j == 1:
                port_ret -= tc 

            daily_ret_records.append({'date': date, 'factor_return': port_ret})

            # Drifting weights between rebalances
            w = w * (1 + day_ret.reindex(w.index, fill_value=0))
            wsum = w.sum()
            if wsum > 0:
                w = w / wsum

        prev_weights = target_weights

    ret_df = pd.DataFrame(daily_ret_records).set_index('date')
    ret_df.index = pd.DatetimeIndex(ret_df.index)

    # Removing data prior to 2013 - need at least three years of data for trailing 3-year average
    ret_df = ret_df.loc['2013-01-01':]
    ret_df = ret_df[~ret_df.index.duplicated(keep='last')]

    bmk = benchmark_returns.reindex(ret_df.index).fillna(0)
    bmk = bmk.loc['2013-01-01':]

    results = pd.DataFrame({
        'factor_index': (1 + ret_df['factor_return']).cumprod(),
        'benchmark': (1 + bmk).cumprod(),
    })
    results['excess'] = results['factor_index'] / results['benchmark']
    return results

In [ ]:
# Building daily returns for a market-cap weighted index of the same universe
def build_mktcap_benchmark_returns(market_cap: pd.DataFrame, price_panel: pd.DataFrame, rebal_dates: pd.DatetimeIndex):
    all_dates = price_panel.index.sort_values()
    daily_records = []

    for i, rebal_date in enumerate(rebal_dates):
        if rebal_date not in all_dates:
            continue
        mc_xsec = market_cap.loc[rebal_date].dropna()
        bench_w = mc_xsec / mc_xsec.sum()

        start_idx = all_dates.get_loc(rebal_date)
        if i + 1 < len(rebal_dates):
            next_rebal = rebal_dates[i + 1]
            if next_rebal in all_dates:
                end_idx = all_dates.get_loc(next_rebal)
            else:
                end_idx = len(all_dates)
        else:
            end_idx = len(all_dates)

        period_dates = all_dates[start_idx:end_idx + 1]
        if len(period_dates) < 2:
            continue

        held = bench_w.index.intersection(price_panel.columns)
        period_prices = price_panel.loc[period_dates, held]
        daily_returns = period_prices.pct_change()

        w = bench_w.reindex(held, fill_value=0)
        w = w / w.sum() if w.sum() > 0 else w

        for date in period_dates[1:]:
            day_ret = daily_returns.loc[date].fillna(0)
            port_ret = (w * day_ret.reindex(w.index, fill_value=0)).sum()
            daily_records.append({'date': date, 'benchmark_return': port_ret})

            w = w * (1 + day_ret.reindex(w.index, fill_value=0))
            wsum = w.sum()
            if wsum > 0:
                w = w / wsum

    bmk_df = pd.DataFrame(daily_records).set_index('date')
    bmk_df.index = pd.DatetimeIndex(bmk_df.index)
    bmk_df = bmk_df[~bmk_df.index.duplicated(keep='last')]
    return bmk_df['benchmark_return']

benchmark_returns = build_mktcap_benchmark_returns(market_cap, close, rebal_dates)

In [ ]:
# Run the backtest
results = run_backtest(weights_history, close, benchmark_returns)

results.tail()

In [ ]:
# Building out performance metrics for SY index and benchmark
def compute_performance_stats(results: pd.DataFrame):
    factor_ret = results['factor_index'].pct_change().dropna()
    bmk_ret = results['benchmark'].pct_change().dropna()
    excess_ret = factor_ret - bmk_ret

    trading_days = 252

    def stats_for(rets):
        cum = (1 + rets).cumprod()
        total_ret = cum.iloc[-1] - 1
        n_years = len(rets) / 252
        ann_ret = (1 + total_ret) ** (1 / n_years) - 1
        ann_vol = rets.std() * np.sqrt(trading_days)
        sharpe = ann_ret / ann_vol if ann_vol > 0 else 0
        max_dd = (cum / cum.cummax() - 1).min()
        return {
            'Annualized Return': ann_ret,
            'Annualized Volatility': ann_vol,
            'Sharpe Ratio': sharpe,
            'Max Drawdown': max_dd,
            'Total Return': total_ret,
        }

    df_stats = pd.DataFrame({
        'Factor Index': stats_for(factor_ret),
        'Benchmark':    stats_for(bmk_ret),
        'Excess':       stats_for(excess_ret),
    })

    tracking_error = excess_ret.std() * np.sqrt(trading_days)
    info_ratio = (excess_ret.mean() * trading_days) / tracking_error if tracking_error > 0 else 0
    df_stats.loc['Information Ratio'] = ['', '', info_ratio]
    df_stats.loc['Tracking Error'] = ['', '', tracking_error]

    return df_stats


stats_df = compute_performance_stats(results)

stats_df

In [ ]:
# Building out turnover statistics
def compute_turnover_stats(weights_history: dict):
    dates = sorted(weights_history.keys())
    turnovers = []

    for i in range(1, len(dates)):
        old_w = weights_history[dates[i - 1]]
        new_w = weights_history[dates[i]]
        all_RICs = old_w.index.union(new_w.index)
        t = (
            new_w.reindex(all_RICs, fill_value=0)
            - old_w.reindex(all_RICs, fill_value=0)
        ).abs().sum() / 2
        turnovers.append(t)

    rebal_per_year = {'Q': 4, 'M': 12, 'A': 1, '6ME': 2}.get(Config.REBALANCE_FREQ)
    avg_turnover = np.mean(turnovers) if turnovers else 0
    annual_tc_drag = avg_turnover * rebal_per_year * Config.TC_BPS / 10000

    return {
        'Mean One-Way Turnover': f'{avg_turnover:.2%}',
        'Rebalance Count':        len(dates),
        'Est. Annual TC Drag':    f'{annual_tc_drag:.4%}',
    }

turnover_stats = compute_turnover_stats(weights_history)
turnover_stats

In [ ]:
# Visualizations to compare performance between the two indices
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 9), height_ratios=[2, 1])
fig.suptitle('Shareholder Yield Index vs Mkt-Cap-Weighted Benchmark',
             fontsize=13, fontweight='bold')

ax1.plot(results.index, results['factor_index'] - 1, label='SY Factor Index',
         linewidth=1.6, color='blue')
ax1.plot(results.index, results['benchmark'] - 1, label='Mkt-Cap Benchmark',
         linewidth=1.6, color='grey', linestyle='--')
ax1.set_ylabel('Growth Pct (%)')
ax1.legend(loc='upper left')
ax1.grid(True, alpha=0.3)

ax2.plot(results.index, results['excess'], label='Factor / Benchmark',
         linewidth=1.5, color='green')
ax2.axhline(y=1.0, color='grey', linestyle=':', linewidth=0.8)
ax2.set_ylabel('Relative Performance')
ax2.set_xlabel('Date')
ax2.legend(loc='upper left')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Visualizations to compare the drawdowns of both indices
factor_dd = results['factor_index'] / results['factor_index'].cummax() - 1
bench_dd = results['benchmark'] / results['benchmark'].cummax() - 1

fig, ax = plt.subplots(figsize=(14, 5))
ax.fill_between(results.index, factor_dd * 100, 0, color='blue', alpha=0.4, label='SY Index')
ax.fill_between(results.index, bench_dd * 100, 0, color='grey', alpha=0.4, label='Benchmark')
ax.set_ylabel('Drawdown (%)')
ax.set_xlabel('Date')
ax.set_title('Drawdown Comparison')
ax.legend(loc='lower left')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Visualizations to compare the rolling 12-month excess return between both indices
factor_ret = results['factor_index'].pct_change()
bench_ret = results['benchmark'].pct_change()
excess_ret = factor_ret - bench_ret

rolling_excess = (1 + excess_ret).rolling(252).apply(np.prod, raw=True) - 1

fig, ax = plt.subplots(figsize=(14, 5))
ax.fill_between(rolling_excess.index, rolling_excess * 100, 0,
                where=(rolling_excess > 0), color='green', alpha=0.5, label='Outperformance')
ax.fill_between(rolling_excess.index, rolling_excess * 100, 0,
                where=(rolling_excess <= 0), color='red', alpha=0.5, label='Underperformance')
ax.axhline(y=0, color='#6b7280', linewidth=0.8)
ax.set_ylabel('Rolling 12-Month Excess Return (%)')
ax.set_xlabel('Date')
ax.set_title('Rolling 12-Month Excess Return: SY Index vs Benchmark')
ax.legend(loc='upper left')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Visualizations to evaluate sector allocations amongst each index
latest_date = list(weights_history.keys())[-1]
latest_w = weights_history[latest_date]

# Factor index sector allocation
factor_sec = latest_w.groupby(sectors.reindex(latest_w.index)).sum().sort_values(ascending=False)

# Benchmark sector allocation on same date
mc_latest = market_cap.loc[latest_date].dropna()
bench_w = mc_latest / mc_latest.sum()
bench_sec = bench_w.groupby(sectors.reindex(bench_w.index)).sum().reindex(factor_sec.index)

sector_df = pd.DataFrame({
    'Factor Index': factor_sec,
    'Benchmark':    bench_sec,
}).fillna(0)
sector_df['Active Weight'] = sector_df['Factor Index'] - sector_df['Benchmark']

fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(sector_df))
width = 0.35
ax.bar(x - width/2, sector_df['Factor Index'] * 100, width, label='SY Index', color='blue')
ax.bar(x + width/2, sector_df['Benchmark'] * 100, width, label='Benchmark', color='grey')
ax.set_xticks(x)
ax.set_xticklabels(sector_df.index, rotation=45, ha='right')
ax.set_ylabel('Weight (%)')
ax.set_title(f'Sector Allocation: SY Index vs Benchmark')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()